In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv('SH_ID') 
client_secret = os.getenv('SH_SECRET') 

print("Pobieranie tokenu dostępu...")
auth_response = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret
    }
)
token = auth_response.json().get("access_token")

url = "https://sh.dataspace.copernicus.eu/api/v1/statistics"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json"
}

evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B04", "B08", "SCL", "dataMask"],
    output: [
      { id: "ndvi", bands: 1 },
      { id: "dataMask", bands: 1 }
    ]
  };
}

function evaluatePixel(samples) {
  // Ignorujemy chmury i cienie (klasy SCL: 3, 8, 9, 10)
  if ([3, 8, 9, 10].includes(samples.SCL) || samples.dataMask === 0) {
      return { ndvi: [NaN], dataMask: [0] };
  }
  let ndvi = (samples.B08 - samples.B04) / (samples.B08 + samples.B04);
  return { ndvi: [ndvi], dataMask: [1] };
}
"""
payload = {
  "input": {
    "bounds": {
      "bbox": [16.90, 51.05, 17.15, 51.15]
    },
    "data": [
      {
        "type": "sentinel-2-l2a",
        "dataFilter": { "mosaickingOrder": "leastCC" }
      }
    ]
  },
  "aggregation": {
    "timeRange": {
      "from": "2025-10-01T00:00:00Z", 
      "to": "2026-04-24T00:00:00Z"
    },
    "aggregationInterval": { "of": "P5D" }, 
    "evalscript": evalscript,
    "resx": 0.0005,
    "resy": 0.0005
  }
}

print("Przeliczanie historii NDVI w chmurze CDSE. To potrwa kilka sekund...")
response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    data = response.json()
    wyniki = []
    
    for item in data['data']:
        data_odczytu = item['interval']['from'][:10]

        statystyki = item['outputs']['ndvi']['bands']['B0']['stats']
        
        if statystyki['sampleCount'] > 0:
            wyniki.append({
                "Data": data_odczytu,
                "Srednie_NDVI": statystyki['mean'],
                "Min_NDVI": statystyki['min'],
                "Max_NDVI": statystyki['max']
            })
            
    df = pd.DataFrame(wyniki)
    print("✅ SUKCES! Pobrano szereg czasowy.")
    display(df.tail())
else:
    print(f"❌ BŁĄD {response.status_code}: {response.text}")

Pobieranie tokenu dostępu...
Przeliczanie historii NDVI w chmurze CDSE. To potrwa kilka sekund...
✅ SUKCES! Pobrano szereg czasowy.


,Data,Srednie_NDVI,Min_NDVI,Max_NDVI
36,2026-03-30,0.323108,-1.0,0.962797
37,2026-04-04,0.333479,-1.0,0.930364
38,2026-04-09,0.370325,-1.0,1.0
39,2026-04-14,0.28127,-1.0,0.889046
40,2026-04-19,0.422123,-1.0,1.0
